# Perceptron lab partial solutions

This notebook contains example partial solutions to the preceptron lab in Python. There are many equally valid ways to implement this, and the coding style is somewhat arbitrary. Students are free to select their preferred implementation language and design patterns.

### Configure environment

In [83]:
import numpy as np
from pylab import *

# get load_points function and load points
from load  import * 
points=load_points("points.txt")

# Let's say 2 sigfigs today, for readability
set_printoptions(formatter={'float': lambda x: "{0:0.2f}".format(x)})

# Firing-rate nonlinearity:
# Heaviside with threshold value arbitrarily chosen as 1
# (you may make a different choice)
Θ = lambda x:heaviside(x,1)

### Set up problem

Currently, `points` is a list of tuples, where each tuple contains a string (the "red", "blue" label), and a length-5 vector of point coordinates.
Some folks might choose to reformat this slightly for computational convenience. 

In [84]:
# Extracting data into numeric arrays for numpy
labels, values = zip(*points)
y = float32([1 if l=='red' else 0 for l in labels])
x = float32(values).T
Ndim,Npoints = shape(x)

## Implement the perceptron learning rule

### Online (one sample at a time)

In [35]:
η = 0.1     # learning rate (with normalization factor folded in)
Nepoch = 20 # How long to train
w = randn(Ndim)*.1 # Initial weights
for i in range(Nepoch):
    for j in permutation(Npoints):
        a  = w@x[:,j]  # synaptic activation
        yh = Θ(a) # spiking output
        ε  = y[j]-yh # error signal
        Δw = x[:,j]*ε  # Plasticity update
        w += η*Δw
    print("\repoch %03d %03d%% correct"%(i,(1-mean(abs(ε)))*100),
          end='',flush=True)
print()
print('weights',w)

epoch 019 100% correct
weights [0.49 -2.01 0.87 -2.01 1.29]


### Batch (averaged) version

In [36]:
η = 0.1/Npoints    # learning rate (with normalization factor folded in)
Nepoch = 20        # How long to train
w = randn(Ndim)*.1 # Initial weights
for i in range(Nepoch):
    a  = w@x  # synaptic activation
    yh = Θ(a) # spiking output
    ε  = y-yh # error signal
    Δw = x@ε  # Plasticity update
    w += η*Δw
    print("\repoch %03d %03d%% correct"%(i,(1-mean(abs(ε)))*100),
          end='',flush=True)
print()
print('weights',w)

epoch 019 100% correct
weights [0.13 -0.23 0.33 -0.44 0.14]


## Open-ended explorations 

### Here's a more compact style that adds a threshold

In [37]:
x1 = concatenate([x,ones((1,Npoints))],0) # Make the last feature a threshold
w1 = randn(Ndim+1)*.1
for i in range(Nepoch):
    w1 += η*(x1@(y-Θ(w1@x1))) 
print("%03d%% correct"%((1-mean(abs(y-Θ(w1@x1))))*100))
print('weights, threshold:',w1)

100% correct
weights, threshold: [0.18 -0.28 0.36 -0.49 0.13 -0.13]


### A different nonlinearity? 

If we use a logistic firing-rate nonlinearity, the delta rule implements logistic regression (binary classification). 

*Don't worry about the specifics of "cross entropy loss" here, this is outside the SEMT30003 curriculum. It's provided here as an example of something a student migh encounter when exploring the topic further, possibly connecting it to other things they have been learning.*

In [38]:
σ  = lambda x:1/(1+exp(-x)) # logistic sigmoid
wb = randn(Ndim+1)*.1
for i in range(Nepoch):
    wb += η*(x1@(y-σ(wb@x1)))
    a = wb@x1
    l = logaddexp(0,-a)+(1-y)*a
    print("%02d: Cross-entropy loss:"%i,mean(l))
print("%03d%% correct (using old hard threshold)"%((1-mean(abs(y-Θ(wb@x1))))*100))
print('weights, threshold:',wb)

00: Cross-entropy loss: 0.47050854453797636
01: Cross-entropy loss: 0.372165419571268
02: Cross-entropy loss: 0.31291026348339857
03: Cross-entropy loss: 0.27370629663881135
04: Cross-entropy loss: 0.2458872037322748
05: Cross-entropy loss: 0.22511238099951444
06: Cross-entropy loss: 0.20896180414438814
07: Cross-entropy loss: 0.195989353547309
08: Cross-entropy loss: 0.18529401441728605
09: Cross-entropy loss: 0.17628798022442918
10: Cross-entropy loss: 0.16857132284210458
11: Cross-entropy loss: 0.16186219921797507
12: Cross-entropy loss: 0.15595619245330353
13: Cross-entropy loss: 0.1507014392462868
14: Cross-entropy loss: 0.14598274407901096
15: Cross-entropy loss: 0.1417111225464194
16: Cross-entropy loss: 0.13781672676144838
17: Cross-entropy loss: 0.13424394434167747
18: Cross-entropy loss: 0.1309479305944762
19: Cross-entropy loss: 0.12789210798437817
095% correct (using old hard threshold)
weights, threshold: [0.11 -0.37 0.42 -0.60 0.24 -0.08]


### What if some examples are bad? 

It shouldn't be possible to get 100% training accuracy

In [39]:
z = copy(y)
Nflip = 3
z[10-Nflip:10+Nflip] = 1-z[10-Nflip:10+Nflip]
w1 = randn(Ndim+1)*.1
for i in range(Nepoch):
    w1 += η*(x1@(z-Θ(w1@x1)))
print("%03d%% correct"%((1-mean(abs(z-Θ(w1@x1))))*100))
print('weights, threshold:',w1)

080% correct
weights, threshold: [-0.02 -0.28 0.42 -0.15 0.06 -0.10]


### (many more explorations possible) 